# 데이터 라벨링 

In [2]:
!pip install ultralytics

In [3]:
import torch
import os
from PIL import Image
import torchvision.transforms as transforms
import shutil # 파일 이동/삭제용

# YOLO 모델 로드를 위해 ultralytics import
from ultralytics import YOLO

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- 1. 모델 로드 ---
model_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3.pt'

# Ultralytics YOLO 모델 로드
try:
    # YOLO 생성자를 사용하여 모델을 로드합니다.
    # YOLO 모델은 로드 시 자동으로 평가 모드로 설정됩니다.
    model = YOLO(model_path)
    print(f"모델이 성공적으로 로드되었습니다: {model_path}")
except Exception as e:
    print(f"모델 로드 중 오류 발생: {e}")
    print("모델 경로가 올바른지 확인하거나, 모델 파일이 손상되지 않았는지 확인하십시오.")
    # 문제가 계속되면 스크립트 실행을 중단할 수 있습니다。
    # exit()

# --- 2. 이미지 세트 경로 설정 및 출력 디렉토리 생성 ---
# 'image_set' 폴더의 경로입니다. 이전 단계에서 확인한 경로를 사용합니다.
image_set_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/image_set'

# 처리된 이미지를 저장할 새 디렉토리입니다.
# 원본 파일을 삭제하기 전에 이곳에 저장하는 것이 안전합니다.
output_dir = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images'
os.makedirs(output_dir, exist_ok=True)
print(f"처리된 이미지는 다음 위치에 저장됩니다: {output_dir}")

# --- 3. 이미지 전처리 정의 (YOLO 모델은 자체 전처리를 수행하는 경우가 많으므로 현재는 사용하지 않습니다) ---
# 만약 모델이 특정 전처리를 요구한다면 이 부분을 다시 활성화하고 조정해야 합니다.
# preprocess = transforms.Compose([
#     transforms.Resize((224, 224)),  # 모델 입력 크기에 맞게 조정하세요 (예: 224x224)
#     transforms.ToTensor(),          # 이미지를 PyTorch 텐서로 변환
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

# --- 4. 이미지 처리 및 저장 루프 ---
# image_set_path 내의 모든 이미지 파일을 가져옵니다.
image_files = [f for f in os.listdir(image_set_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
print(f"'{image_set_path}'에서 {len(image_files)}개의 이미지를 찾았습니다.")

print("\n!!! 경고: 이 코드는 처리 후 원본 이미지를 삭제합니다. 실행 전에 중요한 파일을 백업하십시오. !!!")
print("계속하려면 이 셀을 실행하세요. 원본 삭제를 원하지 않는다면 `os.remove(original_image_path)` 라인을 주석 처리하세요.")

skipped_images = [] # 건너뛴 이미지 이름을 저장할 리스트
processed_image_count = 0 # 성공적으로 처리된 이미지 개수 (원본 이미지 기준)

for image_name in image_files:
    original_image_path = os.path.join(image_set_path, image_name)
    try:
        image = Image.open(original_image_path).convert('RGB')
        results = model(image, verbose=False)
        base_name = os.path.splitext(image_name)[0]

        # 알약이 정확히 2개 감지된 경우에만 처리
        if results and len(results[0].boxes) == 2:
            detected_boxes = []
            for box_tensor in results[0].boxes.xyxy:
                box = box_tensor.cpu().numpy().astype(int)
                detected_boxes.append(box)

            # x1 좌표를 기준으로 정렬 (가장 왼쪽부터)
            detected_boxes.sort(key=lambda b: b[0])

            # 가장 왼쪽 알약 (front)
            leftmost_box = detected_boxes[0]
            x1, y1, x2, y2 = leftmost_box
            cropped_pill_image = image.crop((x1, y1, x2, y2))
            front_output_filename = f"{base_name}_front.jpg"
            front_output_path = os.path.join(output_dir, front_output_filename)
            cropped_pill_image.save(front_output_path) # JPEG 형식으로 저장
            #print(f"Saved: {front_output_path}")

            # 가장 오른쪽 알약 (back)
            rightmost_box = detected_boxes[-1]
            x1, y1, x2, y2 = rightmost_box
            cropped_pill_image = image.crop((x1, y1, x2, y2))
            back_output_filename = f"{base_name}_back.jpg"
            back_output_path = os.path.join(output_dir, back_output_filename)
            cropped_pill_image.save(back_output_path) # JPEG 형식으로 저장
            #print(f"Saved: {back_output_path}")
            processed_image_count += 1 # 2개의 알약이 성공적으로 처리되었으므로 카운트 증가

            # --- 4. 원본 이미지 삭제 ---
            # 이 단계는 신중하게 수행해야 합니다.
            #os.remove(original_image_path)
            #print(f"Deleted original image: {original_image_path}")

        else:
            # 알약이 2개가 아니면 건너뛰고 경고 메시지 출력
            num_detections = 0
            if results and results[0].boxes:
                num_detections = len(results[0].boxes)
            #print(f"Info: {image_name}에서 {num_detections}개의 알약이 감지되었습니다. 2개가 아니므로 처리를 건너뜁니다.")
            skipped_images.append(image_name) # 건너뛴 이미지 이름 리스트에 추가

    except Exception as e:
        #print(f"이미지 처리 중 오류 발생: {image_name} - {e}")
        skipped_images.append(image_name) # 오류 발생 이미지도 건너뛴 이미지로 간주

print("\n모든 이미지 처리가 완료되었습니다.")

# --- 통계 출력 ---
print("\n--- 이미지 처리 통계 ---")
print(f"총 원본 이미지 수: {len(image_files)}개")
print(f"성공적으로 처리된 원본 이미지 수 (2개 알약 감지): {processed_image_count}개")
print(f"  -> 생성된 알약 이미지 파일 수 (front/back): {processed_image_count * 2}개")
print(f"건너뛴 원본 이미지 수 (2개 알약 미감지 또는 오류): {len(skipped_images)}개")

if skipped_images:
    print(f"\n--- 건너뛴 이미지 목록 ({len(skipped_images)}개) ---")
    for skipped_img in skipped_images:
        print(skipped_img)
else:
    print("\n건너뛴 이미지가 없습니다.")

Using device: cpu
모델이 성공적으로 로드되었습니다: /Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3.pt
처리된 이미지는 다음 위치에 저장됩니다: /Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images
'/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/image_set'에서 3072개의 이미지를 찾았습니다.

!!! 경고: 이 코드는 처리 후 원본 이미지를 삭제합니다. 실행 전에 중요한 파일을 백업하십시오. !!!
계속하려면 이 셀을 실행하세요. 원본 삭제를 원하지 않는다면 `os.remove(original_image_path)` 라인을 주석 처리하세요.

모든 이미지 처리가 완료되었습니다.

--- 이미지 처리 통계 ---
총 원본 이미지 수: 3072개
성공적으로 처리된 원본 이미지 수 (2개 알약 감지): 2906개
  -> 생성된 알약 이미지 파일 수 (front/back): 5812개
건너뛴 원본 이미지 수 (2개 알약 미감지 또는 오류): 166개

--- 건너뛴 이미지 목록 (166개) ---
201204602.png
198801525.png
200902212.png
201205247.png
201204603.png
198000118.png
199201630.png
201200315.png
200502128.png
200902577.png
201205536.png
201207490.png
201105149.png
199200885.png
199501042.png
200905023.png
201103677.png
198901324.png
201104536.png
199600913.png
199600091.png
199601172.png
198801534.png


## csv파일로드

In [8]:
import pandas as pd

# pill_data.csv 파일의 경로
pill_data_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/src/pill_data.csv'

# CSV 파일 로드
try:
    pill_df = pd.read_csv(pill_data_path)
    print("CSV 파일이 성공적으로 로드되었습니다.")
    # 데이터프레임의 첫 5행과 컬럼 정보 출력
    display(pill_df.head())
    pill_df.info()
except FileNotFoundError:
    print(f"오류: '{pill_data_path}' 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
except Exception as e:
    print(f"CSV 파일을 로드하는 중 오류 발생: {e}")

CSV 파일이 성공적으로 로드되었습니다.


/var/folders/bs/std35cqs3_n4nhgz4g2t7nzr0000gn/T/ipykernel_53753/1225826310.py:8: DtypeWarning: Columns (13,14,17) have mixed types. Specify dtype option on import or set low_memory=False.
  pill_df = pd.read_csv(pill_data_path)


,품목일련번호,품목명,업소일련번호,업소명,성상,큰제품이미지,표시앞,표시뒤,의약품제형,색상앞,...,표기이미지앞,표기이미지뒤,표기코드앞,표기코드뒤,변경일자,사업자번호,품목영문명,보험코드,표준코드,Unnamed: 33
0,200808876,가스디알정50밀리그램(디메크로틴산마그네슘),19540006,일동제약(주),녹색의원형필름코팅정,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,IDG,-,원형,연두,...,-,-,-,-,-,6828500435,-,-,8806429038103|8806429038110|8806429038127|8806...,NaN
1,200808877,페라트라정2.5밀리그램(레트로졸),19560004,(주)유한양행,어두운황색의원형필름코팅정,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,YH,LT,원형,노랑,...,-,-,-,-,-,1188100601,-,-,8806421038804|8806421038811|8806421038828,NaN
2,200808948,졸뎀속붕정(졸피뎀타르타르산염),20080422,보령제약(주),흰색의원형구강붕해정제,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,-,-,원형,하양,...,-,-,-,-,-,1348502031,-,-,8806419054205|8806419054212|8806419054229,NaN
3,200809076,가스프렌정(모사프리드시트르산염이수화물),19760002,경동제약(주),분할선을가진흰색의장방형필름코팅정제,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,KD,분할선,장방형,하양,...,-,-,-,-,20211223,1248135518,GasprenTab,648102540,8806481025400|8806481025417|8806481025424|8806...,NaN
4,200809361,바르탄정(발사르탄),19870002,(주)휴온스,엷은적색의원형필름코팅정,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,V분할선T,HS8,원형,분홍,...,-,-,-,-,-,8638700270,-,-,8806706056509|8806706056516|8806706056523,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27616 entries, 0 to 27615
Data columns (total 34 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   품목일련번호          27616 non-null  int64  
 1   품목명             27616 non-null  object 
 2   업소일련번호          27616 non-null  int64  
 3   업소명             27616 non-null  object 
 4   성상              27616 non-null  object 
 5   큰제품이미지          27616 non-null  object 
 6   표시앞             27615 non-null  object 
 7   표시뒤             27614 non-null  object 
 8   의약품제형           27616 non-null  object 
 9   색상앞             27616 non-null  object 
 10  색상뒤             27616 non-null  object 
 11  분할선앞            27616 non-null  object 
 12  분할선뒤            27616 non-null  object 
 13  크기장축            27616 non-null  object 
 14  크기단축            27616 non-null  object 
 15  크기두께            27616 non-null  object 
 16  이미지생성일자(약학정보원)  27616 non-null  int64  
 17  분류번호            27616 non-null 

## 모든 처리된 이미지 파일에서 메타데이터 추출 및 JSON으로 저장
이제 processed_images 디렉토리에 있는 모든 _front.jpg 및 _back.jpg 파일에 대해 메타데이터를 추출하고, 이를 하나의 JSON 파일로 저장하는 코드를 작성합니다. 각 파일명에서 일련번호와 앞/뒤 구분을 추출하여 pill_df에서 해당 정보를 조회합니다.

In [9]:
import os
import json
import pandas as pd

# processed_images 폴더 경로 (이전 셀에서 정의됨)
processed_images_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images'

# 메타데이터를 저장할 리스트
all_pills_metadata = []

# processed_images 폴더의 모든 파일 목록 가져오기
processed_image_files = [f for f in os.listdir(processed_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"'{processed_images_path}'에서 {len(processed_image_files)}개의 처리된 이미지를 찾았습니다.\n")

for image_filename in processed_image_files:
    full_image_path = os.path.join(processed_images_path, image_filename)

    # 파일명에서 확장자 제거
    base_name = os.path.splitext(image_filename)[0]

    serial_number_str = None
    part_type = None

    # _front 또는 _back 구분자와 일련번호 추출
    if '_front' in base_name:
        serial_number_str = base_name.replace('_front', '')
        part_type = 'front'
    elif '_back' in base_name:
        serial_number_str = base_name.replace('_back', '')
        part_type = 'back'
    else:
        # _front/_back 구분이 없는 경우, 일련번호만 사용
        serial_number_str = base_name
        part_type = 'unknown'

    if not serial_number_str.isdigit():
        print(f"경고: '{image_filename}' 파일명에서 유효한 일련번호를 추출할 수 없습니다. 스킵합니다.")
        continue

    example_serial_number = int(serial_number_str)

    # 일련번호에 해당하는 행 찾기
    pill_metadata_row = pill_df[pill_df['품목일련번호'] == example_serial_number]

    if not pill_metadata_row.empty:
        extracted_data = {
            'image_filename': image_filename,
            'serial_number': serial_number_str,
            'part_type': part_type,
            'metadata': {}
        }

        # part_type에 따라 추출할 컬럼 목록 정의 및 새 키 이름 매핑
        if part_type == 'front':
            target_columns_map = {
                '표시앞': '표시',
                '색상앞': '색상',
                '분할선앞': '분할선',
                '의약품제형': '제형'
            }
        elif part_type == 'back':
            target_columns_map = {
                '표시뒤': '표시',
                '색상뒤': '색상',
                '분할선뒤': '분할선',
                '의약품제형': '제형'
            }
        else:
            # _front/_back 구분이 없는 경우 (fallback)
            target_columns_map = {
                '표시앞': '표시_앞', '표시뒤': '표시_뒤',
                '색상앞': '색상_앞', '색상뒤': '색상_뒤',
                '분할선앞': '분할선_앞', '분할선뒤': '분할선_뒤',
                '의약품제형': '제형'
            }

        for original_col, new_key in target_columns_map.items():
            if original_col in pill_metadata_row.columns:
                extracted_data['metadata'][new_key] = str(pill_metadata_row[original_col].iloc[0]) # JSON 직렬화를 위해 문자열로 변환
            else:
                extracted_data['metadata'][new_key] = None # 컬럼이 없는 경우 None으로 처리

        all_pills_metadata.append(extracted_data)
        # print(f"Processed: {image_filename}")
    else:
        print(f"경고: 일련번호 {serial_number_str} 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: {image_filename})")

# JSON 파일로 저장
output_json_path = os.path.join(processed_images_path, 'pill_metadata.json')
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(all_pills_metadata, f, ensure_ascii=False, indent=4)

print(f"\n모든 메타데이터가 '{output_json_path}'에 JSON 형식으로 저장되었습니다.")
print(f"총 {len(all_pills_metadata)}개의 이미지 파일의 메타데이터가 처리되었습니다.")

'/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images'에서 5812개의 처리된 이미지를 찾았습니다.

경고: 일련번호 199304303 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199304303_back.jpg)
경고: 일련번호 198300433 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 198300433_back.jpg)
경고: 일련번호 199801864 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199801864_front.jpg)
경고: 일련번호 200800505 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 200800505_back.jpg)
경고: 일련번호 199300145 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199300145_back.jpg)
경고: 일련번호 199001165 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199001165_back.jpg)
경고: 일련번호 199701146 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199701146_back.jpg)
경고: 일련번호 199300354 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199300354_back.jpg)
경고: 일련번호 198400648 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 198400648_back.jpg)
경고: 일련번호 198900344 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 198900344_back.jpg)
경고: 일련번호 199900547 에 해당하는 약품을 pill_df에서 찾을 수 없습니다. (파일: 199900547_front.jpg)
경고: 일련번호 198800284 에 해당하는 약품을 pill_df에서 찾을

## 생성된 JSON 파일 내용 확인
pill_metadata.json 파일의 내용을 읽어와 첫 5개의 항목을 출력하여 VLM 파인튜닝에 사용될 메타데이터의 구조를 확인

In [11]:
import json
import os

# processed_images 폴더 경로 (이전 셀에서 정의됨)
processed_images_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images'

# JSON 파일 경로
output_json_path = os.path.join(processed_images_path, 'pill_metadata.json')

try:
    with open(output_json_path, 'r', encoding='utf-8') as f:
        metadata_list = json.load(f)

    print(f"'{output_json_path}'에서 로드된 총 메타데이터 항목 수: {len(metadata_list)}\n")

    # 첫 5개 항목만 출력하여 구조 확인
    print("--- 첫 5개 메타데이터 항목 ---")
    for i, item in enumerate(metadata_list[:5]):
        print(f"항목 {i+1}:")
        print(json.dumps(item, ensure_ascii=False, indent=4))
        print("---")

except FileNotFoundError:
    print(f"오류: '{output_json_path}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"JSON 파일을 읽는 중 오류 발생: {e}")

'/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/processed_images/pill_metadata.json'에서 로드된 총 메타데이터 항목 수: 5550

--- 첫 5개 메타데이터 항목 ---
항목 1:
{
    "image_filename": "199301674_front.jpg",
    "serial_number": "199301674",
    "part_type": "front",
    "metadata": {
        "표시": "AFD",
        "색상": "노랑",
        "분할선": "-",
        "제형": "원형"
    }
}
---
항목 2:
{
    "image_filename": "200401859_front.jpg",
    "serial_number": "200401859",
    "part_type": "front",
    "metadata": {
        "표시": "NGP",
        "색상": "자주",
        "분할선": "-",
        "제형": "타원형"
    }
}
---
항목 3:
{
    "image_filename": "201305824_back.jpg",
    "serial_number": "201305824",
    "part_type": "back",
    "metadata": {
        "표시": "-",
        "색상": "-",
        "분할선": "-",
        "제형": "원형"
    }
}
---
항목 4:
{
    "image_filename": "197300013_back.jpg",
    "serial_number": "197300013",
    "part_type": "back",
    "metadata": {
        "표시": "BM분할선EU",
        "색상": "-",
 